# Recognize handwritten digits with an MLP

The parts follow the module: data, one neuron and one layer, the network, training, a check of the math by hand, the mistakes, and the learning rate. Every part ends with a self-check that tells you whether it is right.

## Part 1 (coding task): Load MNIST and flatten the images

Load MNIST and look at one image. Running this downloads MNIST the first time, and Parts 4 and 7 train over all 60,000 images, which takes a few minutes on an ordinary laptop. Then turn every image into a row of 784 numbers between 0 and 1, which is the form the network takes as input.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torchvision import datasets

train = datasets.MNIST("data", train=True, download=True)
test = datasets.MNIST("data", train=False, download=True)

print(train.data.shape, train.targets.shape)   # the training images and their labels
print(test.data.shape)                         # the images held back for testing

# One image is a 28 x 28 grid of whole numbers: 0 is blank paper, 255 is full ink.
image = test.data[0]
plt.imshow(image, cmap="gray_r")
plt.title(f"label: {test.targets[0].item()}")
plt.axis("off")
plt.show()

torch.set_printoptions(linewidth=120)
print(image[7:15, 14:22])   # the upper right of that digit, as numbers

In [ ]:
# TODO: flatten every 28 x 28 image into one row of 784 numbers, convert to
#       float, and divide by 255 so each pixel is between 0 and 1.
X_train = ...
X_test = ...
y_train = train.targets
y_test = test.targets

# Self-check.
for _name, _val in [("X_train", X_train), ("X_test", X_test)]:
    assert _val is not Ellipsis, f"`{_name}` is still a `...` in this cell"
assert X_train.shape == (len(train), 784), f"X_train should be ({len(train)}, 784), got {tuple(X_train.shape)}"
assert X_test.shape == (len(test), 784), f"X_test should be ({len(test)}, 784), got {tuple(X_test.shape)}"
assert X_train.dtype == torch.float32, "convert to float before dividing"
assert X_train.min() == 0 and X_train.max() == 1, "pixels should run from 0 to 1"
print("checks passed")

## Part 2 (coding task): One neuron and one layer by hand

Compute one neuron by hand: inputs 1.0, 0.5, 0.0, weights 3, -2, 1 and bias -1. Then take an `nn.Linear(784, 16)` layer, which holds the weights of 16 neurons, and reproduce what it computes on a real image with one matrix multiplication.

In [ ]:
def sigmoid(z):
    # TODO: 1 / (1 + e^(-z)). Use torch.exp so it also works on tensors.
    ...


a = torch.tensor([1.0, 0.5, 0.0])    # inputs
w = torch.tensor([3.0, -2.0, 1.0])   # one weight per input
b = -1.0                             # bias

# TODO: the weighted sum of the inputs plus the bias, then its sigmoid.
z = ...
out = ...

# Self-check.
assert z is not Ellipsis and out is not Ellipsis, "`z` or `out` is still a `...`"
assert out is not None, "sigmoid() returned None, so it is missing a `return`"
assert abs(float(z) - 1.0) < 1e-6, f"the weighted sum should be 1.0, got {float(z):.4f}"
assert abs(float(out) - 0.7311) < 1e-3, f"sigmoid(1) is 0.7311, got {float(out):.4f}"
print(f"weighted sum {float(z):.2f}, output {float(out):.4f}")
print("checks passed")

In [ ]:
# A layer is many neurons reading the same inputs. nn.Linear(784, 16) holds
# 16 neurons' weights as a 16 x 784 matrix, one row per neuron, plus 16 biases.
# They start random: nothing has been learned yet.
torch.manual_seed(0)
layer = nn.Linear(784, 16)
W, bias = layer.weight.detach(), layer.bias.detach()
print(W.shape, bias.shape)
x = X_test[0]                        # the image from Part 1, as 784 numbers

# TODO: all 16 outputs at once, sigmoid(W x + bias). `@` multiplies a matrix
#       by a vector.
layer_out = ...

# Self-check.
assert layer_out is not Ellipsis, "`layer_out` is still a `...`"
assert layer_out.shape == (16,), f"16 neurons give 16 outputs, got {tuple(layer_out.shape)}"
# Row 3 of the matrix form is neuron 3 computed on its own.
assert torch.isclose(layer_out[3], sigmoid((W[3] * x).sum() + bias[3])), "row 3 does not match neuron 3"
# And the layer object computes the same thing.
assert torch.allclose(layer_out, sigmoid(layer(x)), atol=1e-6), "this should match sigmoid(layer(x))"
print(layer_out)
print("checks passed")

## Part 3 (coding task): Build the network

Build the network as an `nn.Module`: 784 inputs, two hidden layers of 16 sigmoid neurons, and 10 outputs. Count its parameters, then measure its accuracy on the test set before any training.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: three nn.Linear layers, 784 -> 16, 16 -> 16 and 16 -> 10.
        self.fc1 = ...
        self.fc2 = ...
        self.fc3 = ...

    def forward(self, x):
        # TODO: sigmoid after the first two layers, nothing after the last.
        #       The 10 outputs are raw scores, one per digit.
        ...


torch.manual_seed(0)
model = MLP()
assert any(isinstance(m, nn.Linear) for m in model.modules()), "the model needs its three nn.Linear layers"
assert model(X_test[:5]) is not None, "forward returned None, so it is missing a `return`"

n_params = sum(p.numel() for p in model.parameters())
print("parameters:", n_params)


def accuracy(model):
    """Fraction of the test images the model gets right."""
    with torch.no_grad():
        scores = model(X_test)       # one row per test image, 10 scores each
        # TODO: the prediction for an image is the position of its highest
        #       score. Return the fraction of predictions that equal y_test,
        #       as a Python float.
        ...


acc_before = accuracy(model)
assert acc_before is not None, "accuracy() returned None, so it is missing a `return`"
print(f"accuracy before training: {acc_before:.4f}")

# Self-check.
assert n_params == 13002, f"expected 13,002 parameters (12,560 + 272 + 170), got {n_params}"
assert model(X_test[:5]).shape == (5, 10), "5 images should give 5 rows of 10 scores"
assert 0.03 < acc_before < 0.25, f"random weights should score near 0.10, got {acc_before:.4f}"
print("checks passed")

## Part 4 (coding task): Train it

Train the network with cross-entropy loss and gradient descent: learning rate 0.5, batches of 32 images, 10 epochs. Print the test accuracy after each epoch. It should end near 0.95.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

for epoch in range(10):
    order = torch.randperm(len(X_train))            # a fresh shuffle every epoch
    for i in range(0, len(X_train), 32):
        batch = order[i:i + 32]
        xb, yb = X_train[batch], y_train[batch]
        # TODO: one training step on this batch: scores, loss, clear the old
        #       gradients, backpropagate, update the weights. The module on
        #       training loops explains each line.
        ...
    print(f"epoch {epoch + 1:2d}  test accuracy {accuracy(model):.4f}")

# Self-check.
acc_after = accuracy(model)
assert acc_after > 0.93, f"expected about 0.95 after 10 epochs, got {acc_after:.4f}"
print("checks passed")

## Part 5 (coding task): Check the math by hand

Check the math by hand. Take the trained weights out of the model and push the first test image through them one layer at a time. Your 10 scores must match the model's.

In [ ]:
W1, b1, W2, b2, W3, b3 = [p.detach() for p in model.parameters()]
print(W1.shape, b1.shape)   # 16 neurons, each with 784 weights and one bias
print(W2.shape, b2.shape)
print(W3.shape, b3.shape)

x = X_test[0]
# TODO: a1 = sigmoid(W1 x + b1), a2 = sigmoid(W2 a1 + b2), scores = W3 a2 + b3.
a1 = ...
a2 = ...
scores = ...

# Self-check.
for _name, _val in [("a1", a1), ("a2", a2), ("scores", scores)]:
    assert _val is not Ellipsis, f"`{_name}` is still a `...` in this cell"
assert torch.allclose(scores, model(x), atol=1e-5), "your scores differ from the model's"
print("scores:", [round(s, 2) for s in scores.tolist()])
print("prediction:", scores.argmax().item(), "  label:", y_test[0].item())
print("checks passed")

## Part 6 (coding task): The mistakes

Find every test image the network gets wrong. Print how many there are and show the first ten, each with its true label and what the network read it as.

In [ ]:
with torch.no_grad():
    predictions = model(X_test).argmax(dim=1)

# TODO: a 1-D tensor holding the index of every test image whose prediction
#       differs from its label.
wrong = ...
assert wrong is not Ellipsis, "`wrong` is still a `...`"
print(len(wrong), "mistakes out of", len(y_test))

fig, axes = plt.subplots(1, 10, figsize=(13, 1.9))
for ax, i in zip(axes, wrong[:10].tolist()):
    ax.imshow(test.data[i], cmap="gray_r")
    ax.set_title(f"{y_test[i].item()} read as {predictions[i].item()}", fontsize=9)
    ax.axis("off")
plt.show()

# Self-check.
assert wrong.dim() == 1, "`wrong` should be a 1-D tensor of indices"
assert len(wrong) == round((1 - accuracy(model)) * len(y_test)), "the count does not match the accuracy"
assert (predictions[wrong] != y_test[wrong]).all(), "some of these were classified correctly"
print("checks passed")

## Part 7 (coding task): Three learning rates

The learning rate sets how far each weight moves per step. Train a fresh network for 3 epochs at each of three learning rates, 0.001, 0.5 and 20, and compare the test accuracy.

In [ ]:
def train(lr, epochs=3):
    """A fresh 784-16-16-10 network trained at this learning rate. Returns its test accuracy."""
    torch.manual_seed(0)
    # TODO: build the same network as Part 3 and train it the way Part 4 does,
    #       using `lr` and `epochs`. Return its test accuracy.
    ...


results = {lr: train(lr) for lr in (0.001, 0.5, 20)}
for lr, acc in results.items():
    assert acc is not None, "train() returned None, so it is missing a `return`"
    print(f"learning rate {lr:<6} test accuracy {acc:.4f}")

# Self-check.
assert results[0.5] > 0.9, "0.5 should pass 0.9 within 3 epochs"
assert results[0.001] < 0.5, "0.001 moves too slowly to learn much in 3 epochs"
assert results[20] < 0.5, "20 overshoots and never settles"
print("checks passed")

## Solution

### Part 1: Load MNIST and flatten the images

In [ ]:
X_train = train.data.reshape(-1, 784).float() / 255
X_test = test.data.reshape(-1, 784).float() / 255
y_train = train.targets
y_test = test.targets

# Self-check.
assert X_train.shape == (len(train), 784), f"X_train should be ({len(train)}, 784), got {tuple(X_train.shape)}"
assert X_test.shape == (len(test), 784), f"X_test should be ({len(test)}, 784), got {tuple(X_test.shape)}"
assert X_train.dtype == torch.float32, "convert to float before dividing"
assert X_train.min() == 0 and X_train.max() == 1, "pixels should run from 0 to 1"
print("checks passed")

# reshape(-1, 784) keeps one row per image and lets PyTorch work out the row
# count. The pixels are stored as whole numbers, so .float() has to come before
# the division. The labels stay as they are: one whole number, 0 to 9, per image.

### Part 2: One neuron and one layer by hand

In [ ]:
def sigmoid(z):
    return 1 / (1 + torch.exp(-z))


a = torch.tensor([1.0, 0.5, 0.0])    # inputs
w = torch.tensor([3.0, -2.0, 1.0])   # one weight per input
b = -1.0                             # bias

z = (w * a).sum() + b
out = sigmoid(z)

# Self-check.
assert abs(float(z) - 1.0) < 1e-6, f"the weighted sum should be 1.0, got {float(z):.4f}"
assert abs(float(out) - 0.7311) < 1e-3, f"sigmoid(1) is 0.7311, got {float(out):.4f}"
print(f"weighted sum {float(z):.2f}, output {float(out):.4f}")
print("checks passed")

# 3(1.0) - 2(0.5) + 1(0.0) - 1 = 1, and sigmoid(1) = 0.7311. The third input is
# 0, so its weight has no effect: a neuron cannot hear an input that is off.

In [ ]:
# A layer is many neurons reading the same inputs. nn.Linear(784, 16) holds
# 16 neurons' weights as a 16 x 784 matrix, one row per neuron, plus 16 biases.
# They start random: nothing has been learned yet.
torch.manual_seed(0)
layer = nn.Linear(784, 16)
W, bias = layer.weight.detach(), layer.bias.detach()
print(W.shape, bias.shape)
x = X_test[0]                        # the image from Part 1, as 784 numbers

layer_out = sigmoid(W @ x + bias)

# Self-check.
assert layer_out.shape == (16,), f"16 neurons give 16 outputs, got {tuple(layer_out.shape)}"
# Row 3 of the matrix form is neuron 3 computed on its own.
assert torch.isclose(layer_out[3], sigmoid((W[3] * x).sum() + bias[3])), "row 3 does not match neuron 3"
# And the layer object computes the same thing.
assert torch.allclose(layer_out, sigmoid(layer(x)), atol=1e-6), "this should match sigmoid(layer(x))"
print(layer_out)
print("checks passed")

# W @ x does 16 weighted sums in one call. Row j of W holds neuron j's weights,
# so entry j of the result is neuron j's weighted sum. layer(x) computes the
# same W x + bias, which is all an nn.Linear does.

### Part 3: Build the network

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 16)
        self.fc2 = nn.Linear(16, 16)
        self.fc3 = nn.Linear(16, 10)

    def forward(self, x):
        a1 = sigmoid(self.fc1(x))
        a2 = sigmoid(self.fc2(a1))
        return self.fc3(a2)


torch.manual_seed(0)
model = MLP()

n_params = sum(p.numel() for p in model.parameters())
print("parameters:", n_params)


def accuracy(model):
    """Fraction of the test images the model gets right."""
    with torch.no_grad():
        scores = model(X_test)       # one row per test image, 10 scores each
        predictions = scores.argmax(dim=1)
        return (predictions == y_test).float().mean().item()


acc_before = accuracy(model)
print(f"accuracy before training: {acc_before:.4f}")

# Self-check.
assert n_params == 13002, f"expected 13,002 parameters (12,560 + 272 + 170), got {n_params}"
assert model(X_test[:5]).shape == (5, 10), "5 images should give 5 rows of 10 scores"
assert 0.03 < acc_before < 0.25, f"random weights should score near 0.10, got {acc_before:.4f}"
print("checks passed")

# Each Linear holds a weight for every input of every neuron, plus one bias per
# neuron: 784*16 + 16 = 12,560, then 16*16 + 16 = 272, then 16*10 + 10 = 170.
# forward is the three layer equations in order. With random weights the
# network answers 9 for every image, which is right about 1 time in 10.

### Part 4: Train it

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

for epoch in range(10):
    order = torch.randperm(len(X_train))            # a fresh shuffle every epoch
    for i in range(0, len(X_train), 32):
        batch = order[i:i + 32]
        xb, yb = X_train[batch], y_train[batch]
        logits = model(xb)                   # 1. the network's scores
        loss = loss_fn(logits, yb)           # 2. how wrong they are
        optimizer.zero_grad()                # 3. clear the old gradients
        loss.backward()                      # 4. gradient for every weight
        optimizer.step()                     # 5. move every weight a little
    print(f"epoch {epoch + 1:2d}  test accuracy {accuracy(model):.4f}")

# Self-check.
acc_after = accuracy(model)
assert acc_after > 0.93, f"expected about 0.95 after 10 epochs, got {acc_after:.4f}"
print("checks passed")

# In batches of 32, each epoch is len(X_train) // 32 weight updates, and most of
# the gain comes in the first epoch.

### Part 5: Check the math by hand

In [ ]:
W1, b1, W2, b2, W3, b3 = [p.detach() for p in model.parameters()]
print(W1.shape, b1.shape)   # 16 neurons, each with 784 weights and one bias
print(W2.shape, b2.shape)
print(W3.shape, b3.shape)

x = X_test[0]
a1 = sigmoid(W1 @ x + b1)
a2 = sigmoid(W2 @ a1 + b2)
scores = W3 @ a2 + b3

# Self-check.
assert torch.allclose(scores, model(x), atol=1e-5), "your scores differ from the model's"
print("scores:", [round(s, 2) for s in scores.tolist()])
print("prediction:", scores.argmax().item(), "  label:", y_test[0].item())
print("checks passed")

# The model is nothing more than these three lines. nn.Linear stores W and b,
# and calling the model runs the same multiplications.

### Part 6: The mistakes

In [ ]:
with torch.no_grad():
    predictions = model(X_test).argmax(dim=1)

wrong = torch.where(predictions != y_test)[0]
print(len(wrong), "mistakes out of", len(y_test))

fig, axes = plt.subplots(1, 10, figsize=(13, 1.9))
for ax, i in zip(axes, wrong[:10].tolist()):
    ax.imshow(test.data[i], cmap="gray_r")
    ax.set_title(f"{y_test[i].item()} read as {predictions[i].item()}", fontsize=9)
    ax.axis("off")
plt.show()

# Self-check.
assert wrong.dim() == 1, "`wrong` should be a 1-D tensor of indices"
assert len(wrong) == round((1 - accuracy(model)) * len(y_test)), "the count does not match the accuracy"
assert (predictions[wrong] != y_test[wrong]).all(), "some of these were classified correctly"
print("checks passed")

# Many of the misses are digits a person would hesitate over too. Count the
# (true, predicted) pairs among the misses to see which two digits this
# network mixes up.

### Part 7: Three learning rates

In [ ]:
def train(lr, epochs=3):
    """A fresh 784-16-16-10 network trained at this learning rate. Returns its test accuracy."""
    torch.manual_seed(0)
    net = MLP()
    opt = torch.optim.SGD(net.parameters(), lr=lr)
    for epoch in range(epochs):
        order = torch.randperm(len(X_train))
        for i in range(0, len(X_train), 32):
            batch = order[i:i + 32]
            loss = loss_fn(net(X_train[batch]), y_train[batch])
            opt.zero_grad()
            loss.backward()
            opt.step()
    return accuracy(net)


results = {lr: train(lr) for lr in (0.001, 0.5, 20)}
for lr, acc in results.items():
    print(f"learning rate {lr:<6} test accuracy {acc:.4f}")

# Self-check.
assert results[0.5] > 0.9, "0.5 should pass 0.9 within 3 epochs"
assert results[0.001] < 0.5, "0.001 moves too slowly to learn much in 3 epochs"
assert results[20] < 0.5, "20 overshoots and never settles"
print("checks passed")

# At 0.001 every step is so small that three epochs barely move the weights.
# At 20 the steps are so large that the weights blow up, the sigmoids get stuck
# at 0 or 1, and the network stays at chance. 0.5 sits between the two.